# Tutorial: SymPy Planar and Fixed-Radius Spherical Shock Normals

This notebook mirrors the two kinematic models used in the project:

- **Planar normal** from three satellites and one bulk velocity vector.
- **Fixed-radius spherical model** from four satellites in the co-moving frame.

It is intentionally minimal: only the equations and code needed to derive and use both models.


## Outline

1. Setup
2. Planar model (symbolic + numeric)
3. Fixed-radius spherical model (symbolic + numeric)
4. Application to one random 4-satellite set


In [1]:
from __future__ import annotations

from datetime import datetime, timedelta, timezone
from pathlib import Path
import json

import numpy as np
import pandas as pd
import sympy as sp
from IPython.display import display, Math


In [2]:
AU_KM = 150_000_000.0
RE_KM = 6371.0
AU_OVER_RE = AU_KM / RE_KM


def unit(vec: np.ndarray) -> np.ndarray:
    vec = np.asarray(vec, dtype=float)
    n = np.linalg.norm(vec)
    if n == 0:
        raise ValueError("Zero-length vector cannot be normalized")
    return vec / n

def render_latex(expr, lhs=None):
    latex_expr = sp.latex(expr)
    if lhs is None:
        display(Math(latex_expr))
    else:
        display(Math(f"{lhs} = {latex_expr}"))


## 1) Planar Model

Equations:

$$
\Delta t_{12} = \frac{t_1 - t_2}{AU / R_e}, \qquad
\Delta t_{13} = \frac{t_1 - t_3}{AU / R_e}
$$

$$
\mathbf{n}\cdot\left(\mathbf{r}_2 - \mathbf{r}_1 - \mathbf{v}\,\Delta t_{12}\right)=0,
\qquad
\mathbf{n}\cdot\left(\mathbf{r}_3 - \mathbf{r}_1 - \mathbf{v}\,\Delta t_{13}\right)=0
$$

with $\|\mathbf{n}\|=1$.


In [3]:
# Symbolic planar solution from the same constraints

u1, u2, u3 = sp.symbols("u1 u2 u3", real=True)
w1, w2, w3 = sp.symbols("w1 w2 w3", real=True)

u = sp.Matrix([u1, u2, u3])
w = sp.Matrix([w1, w2, w3])

n_raw_sym = u.cross(w)
n_hat_sym = sp.simplify(n_raw_sym / sp.sqrt(n_raw_sym.dot(n_raw_sym)))


In [4]:
print("Planar symbolic unit normal:")
render_latex(n_hat_sym, r"\hat{\mathbf{n}}")


Planar symbolic unit normal:


<IPython.core.display.Math object>

In [5]:
def planar_normal(
    r1: np.ndarray,
    r2: np.ndarray,
    r3: np.ndarray,
    t1: datetime,
    t2: datetime,
    t3: datetime,
    v_kms: np.ndarray,
    au_over_re: float = AU_OVER_RE,
) -> dict:
    dt12 = (t1 - t2).total_seconds() / au_over_re
    dt13 = (t1 - t3).total_seconds() / au_over_re

    d12 = np.asarray(r2, dtype=float) - np.asarray(r1, dtype=float) - np.asarray(v_kms, dtype=float) * dt12
    d13 = np.asarray(r3, dtype=float) - np.asarray(r1, dtype=float) - np.asarray(v_kms, dtype=float) * dt13

    n_raw = np.cross(d12, d13)
    n_hat = unit(n_raw)

    return {
        "dt12": dt12,
        "dt13": dt13,
        "d12": d12,
        "d13": d13,
        "n_raw": n_raw,
        "normal": n_hat,
    }


## 2) Fixed-Radius Spherical Model

For 4 satellites, define the co-moving points

$$
\mathbf{p}_i = \mathbf{r}_i - \mathbf{v}\,\Delta t_i,
\qquad
\Delta t_i = \frac{t_i - t_{ref}}{AU / R_e}
$$

and enforce one sphere through all four $\mathbf{p}_i$:

$$
\|\mathbf{p}_i - \mathbf{c}\|^2 = r_0^2, \qquad i=1,2,3,4.
$$

Subtracting equations gives a linear system for center $\mathbf{c}$:

$$
2(\mathbf{p}_1-\mathbf{p}_k)\cdot\mathbf{c} = \|\mathbf{p}_1\|^2 - \|\mathbf{p}_k\|^2,
\qquad k=2,3,4.
$$

Normals at encounter times:

$$
\mathbf{n}_i = \frac{\mathbf{r}_i - (\mathbf{c} + \mathbf{v}\,\Delta t_i)}{\|\mathbf{r}_i - (\mathbf{c} + \mathbf{v}\,\Delta t_i)\|}.
$$


In [6]:
# Symbolic center/radius construction for the fixed-radius sphere model

p1x, p1y, p1z = sp.symbols("p1x p1y p1z", real=True)
p2x, p2y, p2z = sp.symbols("p2x p2y p2z", real=True)
p3x, p3y, p3z = sp.symbols("p3x p3y p3z", real=True)
p4x, p4y, p4z = sp.symbols("p4x p4y p4z", real=True)

p1 = sp.Matrix([p1x, p1y, p1z])
p2 = sp.Matrix([p2x, p2y, p2z])
p3 = sp.Matrix([p3x, p3y, p3z])
p4 = sp.Matrix([p4x, p4y, p4z])

A_sym = 2 * sp.Matrix([
    (p1 - p2).T,
    (p1 - p3).T,
    (p1 - p4).T,
])

b_sym = sp.Matrix([
    p1.dot(p1) - p2.dot(p2),
    p1.dot(p1) - p3.dot(p3),
    p1.dot(p1) - p4.dot(p4),
])

c_sym = A_sym.LUsolve(b_sym)
r0_sym = sp.sqrt((p1 - c_sym).dot(p1 - c_sym))


In [7]:
print("Sphere-through-4-points symbolic linear system:")
render_latex(A_sym, r"\mathbf{A}")
render_latex(b_sym, r"\mathbf{b}")
display(Math(r"\mathbf{c} = \mathbf{A}^{-1}\mathbf{b}"))
display(Math(r"r_0 = \sqrt{(\mathbf{p}_1-\mathbf{c})\cdot(\mathbf{p}_1-\mathbf{c})}"))


Sphere-through-4-points symbolic linear system:


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [8]:
def fixed_radius_sphere_normals(
    sat_data: dict,
    v_kms: np.ndarray,
    au_over_re: float = AU_OVER_RE,
) -> dict:
    names = list(sat_data.keys())
    if len(names) != 4:
        raise ValueError("fixed_radius_sphere_normals expects exactly 4 satellites")

    t_ref = min(sat_data[name]["t"] for name in names)

    dt = {
        name: (sat_data[name]["t"] - t_ref).total_seconds() / au_over_re
        for name in names
    }

    pos = {name: np.asarray(sat_data[name]["r"], dtype=float) for name in names}
    advected = {name: pos[name] - np.asarray(v_kms, dtype=float) * dt[name] for name in names}

    p1, p2, p3, p4 = [advected[name] for name in names]

    A = 2 * np.vstack([
        p1 - p2,
        p1 - p3,
        p1 - p4,
    ])
    b = np.array([
        np.dot(p1, p1) - np.dot(p2, p2),
        np.dot(p1, p1) - np.dot(p3, p3),
        np.dot(p1, p1) - np.dot(p4, p4),
    ])

    if abs(np.linalg.det(A)) < 1e-12:
        raise ValueError("Singular geometry: cannot solve unique sphere")

    center = np.linalg.solve(A, b)
    r0 = float(np.linalg.norm(p1 - center))

    normals = {
        name: unit(pos[name] - (center + np.asarray(v_kms, dtype=float) * dt[name]))
        for name in names
    }

    return {
        "reference_time": t_ref,
        "dt": dt,
        "positions": pos,
        "advected_positions": advected,
        "center": center,
        "radius": r0,
        "normals": normals,
    }


## 3) Application to One Random 4-Satellite Set

We generate one reproducible synthetic event where the data are exactly consistent with the fixed-radius moving-sphere model, then evaluate both solvers.


In [9]:
rng = np.random.default_rng(42)

sat_names = ["ACE", "Wind", "DSCOVR", "MMS1"]
t_ref_true = datetime(2025, 6, 1, 19, 19, 0, tzinfo=timezone.utc)

time_offsets = rng.integers(-240, 241, size=4)
times = {
    name: t_ref_true + timedelta(seconds=int(offset))
    for name, offset in zip(sat_names, time_offsets)
}

v_true_kms = np.array([-420.0, 35.0, -15.0]) + rng.normal(scale=8.0, size=3)
center_true = np.array([140.0, -30.0, 45.0]) + rng.normal(scale=6.0, size=3)
r0_true = 280.0

dir_vectors = rng.normal(size=(4, 3))
dir_vectors = dir_vectors / np.linalg.norm(dir_vectors, axis=1, keepdims=True)

sat_data = {}
true_normals = {}
for name, direction in zip(sat_names, dir_vectors):
    dt_i = (times[name] - t_ref_true).total_seconds() / AU_OVER_RE
    r_i = center_true + v_true_kms * dt_i + r0_true * direction
    sat_data[name] = {"t": times[name], "r": r_i}
    true_normals[name] = direction

summary = pd.DataFrame(
    {
        "sat": sat_names,
        "t": [times[s] for s in sat_names],
        "x_re": [sat_data[s]["r"][0] for s in sat_names],
        "y_re": [sat_data[s]["r"][1] for s in sat_names],
        "z_re": [sat_data[s]["r"][2] for s in sat_names],
    }
)
summary


,sat,t,x_re,y_re,z_re
0,ACE,2025-06-01 19:15:42+00:00,131.829124,-224.527753,244.319541
1,Wind,2025-06-01 19:21:12+00:00,288.699812,-15.510326,273.126404
2,DSCOVR,2025-06-01 19:20:14+00:00,256.099757,-259.245456,141.769660
3,MMS1,2025-06-01 19:18:31+00:00,-73.610663,159.716842,32.398481


In [10]:
triad = sat_names[:3]

planar_result = planar_normal(
    r1=sat_data[triad[0]]["r"],
    r2=sat_data[triad[1]]["r"],
    r3=sat_data[triad[2]]["r"],
    t1=sat_data[triad[0]]["t"],
    t2=sat_data[triad[1]]["t"],
    t3=sat_data[triad[2]]["t"],
    v_kms=v_true_kms,
)

print("Planar triad:", triad)
print("Planar unit normal:", np.round(planar_result["normal"], 6))


Planar triad: ['ACE', 'Wind', 'DSCOVR']
Planar unit normal: [-0.500189  0.459821 -0.733741]


In [11]:
sphere_result = fixed_radius_sphere_normals(sat_data=sat_data, v_kms=v_true_kms)

print("Recovered center:", np.round(sphere_result["center"], 6))
print("True center:     ", np.round(center_true, 6))
print("Recovered r0:", round(sphere_result["radius"], 6))
print("True r0:     ", round(r0_true, 6))


Recovered center: [135.668517 -29.590577  43.359952]
True center:      [132.186923 -29.232958  43.102544]
Recovered r0: 280.0
True r0:      280.0


In [12]:
rows = []
for name in sat_names:
    n_est = sphere_result["normals"][name]
    n_true = true_normals[name]
    alignment = float(np.dot(n_est, n_true))

    p = sphere_result["advected_positions"][name]
    residual = float(np.linalg.norm(p - sphere_result["center"]) - sphere_result["radius"])

    rows.append(
        {
            "sat": name,
            "n_est_x": n_est[0],
            "n_est_y": n_est[1],
            "n_est_z": n_est[2],
            "dot(n_est, n_true)": alignment,
            "sphere_residual_re": residual,
        }
    )

pd.DataFrame(rows)


,sat,n_est_x,n_est_y,n_est_z,"dot(n_est, n_true)",sphere_residual_re
0,ACE,-0.013712,-0.696204,0.717713,1.0,0.0
1,Wind,0.567264,0.048158,0.822127,1.0,0.0
2,DSCOVR,0.447193,-0.821951,0.352726,1.0,0.0
3,MMS1,-0.736813,0.675008,-0.038363,1.0,0.0


## 4) Cross-Validate Python Models vs Wolfram Results

This section reuses the same satellite sets from `Shocks/Kinematic Shock Normals.json`,
pulls satellite states from parquet with Wolfram-like row-selection behavior, and compares
Python model outputs against Wolfram outputs with explicit numeric tolerances.


In [ ]:
SHOCK_PARAMS_PATH = Path("Shocks") / "Shock Params.json"
KINEMATIC_NORMALS_PATH = Path("Shocks") / "Kinematic Shock Normals.json"
DATA_ROOT = Path("Data")
DEFAULT_NAIVE_TZ = "UTC"

shock_params = json.loads(SHOCK_PARAMS_PATH.read_text())
wolfram_normals = json.loads(KINEMATIC_NORMALS_PATH.read_text())

def canonical_sat_name(name: str) -> str:
    return name.lower().replace("_", "").replace("-", "")

def parse_date_value(value, default_naive_tz: str = DEFAULT_NAIVE_TZ) -> pd.Timestamp:
    ts = pd.Timestamp(value)
    if ts.tzinfo is None:
        ts = ts.tz_localize(default_naive_tz)
    return ts.tz_convert("UTC")

def to_standard_rows(df: pd.DataFrame) -> list[dict]:
    std = df.reset_index().copy()

    if "time" not in std.columns:
        std["time"] = pd.NaT

    required = ["time", "X_GSE", "Y_GSE", "Z_GSE"]
    missing = [c for c in required if c not in std.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    return std[required].to_dict("records")

def extract_row_time(row: dict, default_naive_tz: str = DEFAULT_NAIVE_TZ) -> pd.Timestamp:
    return parse_date_value(row["time"], default_naive_tz=default_naive_tz)

def nearest_row_like_wolfram(rows: list[dict], t0: pd.Timestamp, default_naive_tz: str = DEFAULT_NAIVE_TZ) -> dict:
    parsed_times = [extract_row_time(row, default_naive_tz=default_naive_tz) for row in rows]
    valid_idx = [i for i, ts in enumerate(parsed_times) if not pd.isna(ts)]
    if valid_idx:
        deltas = [abs((parsed_times[i] - t0).total_seconds()) for i in valid_idx]
        best = valid_idx[int(np.argmin(deltas))]
    else:
        best = int(np.ceil(len(rows) / 2) - 1)
    return rows[best]

def extract_position_from_row(row: dict) -> np.ndarray:
    pos = np.asarray([row["X_GSE"], row["Y_GSE"], row["Z_GSE"]], dtype=float)
    finite = pos[np.isfinite(pos)]
    if finite.size > 0 and np.max(np.abs(finite)) > 5000:
        pos = pos / RE_KM
    return pos

def load_sat_state_for_t0(event_dir: Path, sat_name: str, t0_raw, default_naive_tz: str = DEFAULT_NAIVE_TZ) -> dict:
    sat_map = {canonical_sat_name(pp.stem): pp for pp in event_dir.glob("*.parquet")}
    sat_file = sat_map[canonical_sat_name(sat_name)]

    rows = to_standard_rows(pd.read_parquet(sat_file))

    t0 = parse_date_value(t0_raw, default_naive_tz=default_naive_tz)
    row_t0 = nearest_row_like_wolfram(rows, t0, default_naive_tz=default_naive_tz)
    row_pos = row_t0

    pos = extract_position_from_row(row_pos)
    used_position_fallback = False
    if not np.isfinite(pos).all():
        rows_with_pos = [row for row in rows if np.isfinite(extract_position_from_row(row)).all()]
        row_pos = nearest_row_like_wolfram(rows_with_pos, t0, default_naive_tz=default_naive_tz)
        pos = extract_position_from_row(row_pos)
        used_position_fallback = True

    t0_row_time = extract_row_time(row_t0, default_naive_tz=default_naive_tz)
    pos_row_time = extract_row_time(row_pos, default_naive_tz=default_naive_tz)

    return {
        "t": t0.to_pydatetime(),
        "r": pos,
        "t0_row_time": t0_row_time,
        "position_row_time": pos_row_time,
        "t0_row_time_offset_s": float((t0_row_time - t0).total_seconds()) if not pd.isna(t0_row_time) else np.nan,
        "position_row_time_offset_s": float((pos_row_time - t0).total_seconds()) if not pd.isna(pos_row_time) else np.nan,
        "used_position_fallback": used_position_fallback,
    }

def build_event_sat_data_wolfram(event: str, sat_names: list[str] | None = None, default_naive_tz: str = DEFAULT_NAIVE_TZ) -> dict:
    event_params = shock_params[event]
    key_map = {canonical_sat_name(k): k for k in event_params}

    if sat_names is None:
        sat_names = list(event_params.keys())

    out = {}
    for sat in sat_names:
        sat_key = key_map[canonical_sat_name(sat)]
        out[sat] = load_sat_state_for_t0(
            event_dir=DATA_ROOT / event,
            sat_name=sat,
            t0_raw=event_params[sat_key]["t0"],
            default_naive_tz=default_naive_tz,
        )
    return out

def unsigned_angle_deg(a: np.ndarray, b: np.ndarray) -> float:
    dot = float(np.clip(np.dot(unit(a), unit(b)), -1.0, 1.0))
    angle = float(np.degrees(np.arccos(dot)))
    return min(angle, 180.0 - angle)



In [ ]:
EVENTS_TO_CHECK = [
    "2022-01-16 19-09-00",
    "2023-02-26 19-23-00",
]

PLANAR_ANGLE_TOL_DEG = 1.0
PLANAR_UNSIGNED_L2_TOL = 2e-2
SPHERE_MAX_NORMAL_ANGLE_TOL_DEG = 1.0
SPHERE_CENTER_TOL_RE = 1e-1
SPHERE_RADIUS_TOL_RE = 1e-1

planar_rows = []
spherical_rows = []
spherical_sat_rows = []

for event in EVENTS_TO_CHECK:
    event_params = shock_params[event]
    event_normals = wolfram_normals[event]
    event_dir = DATA_ROOT / event

    param_key_map = {canonical_sat_name(k): k for k in event_params}

    planar_wf = event_normals.get("planar", {})
    if planar_wf.get("status") == "ok" and planar_wf.get("triplet"):
        triplet = planar_wf["triplet"]
        if all(canonical_sat_name(s) in param_key_map for s in triplet):
            sat_data = build_event_sat_data_wolfram(event=event, sat_names=triplet)

            py_normal = planar_normal(
                r1=sat_data[triplet[0]]["r"],
                r2=sat_data[triplet[1]]["r"],
                r3=sat_data[triplet[2]]["r"],
                t1=sat_data[triplet[0]]["t"],
                t2=sat_data[triplet[1]]["t"],
                t3=sat_data[triplet[2]]["t"],
                v_kms=np.asarray(planar_wf["velocity"], dtype=float),
            )
            wf_normal = np.asarray(planar_wf["normal"], dtype=float)
            py_normal_vec = py_normal["normal"]

            unsigned_angle = unsigned_angle_deg(py_normal_vec, wf_normal)
            unsigned_l2 = float(min(np.linalg.norm(py_normal_vec - wf_normal), np.linalg.norm(py_normal_vec + wf_normal)))

            planar_rows.append({
                "event": event,
                "triplet": ", ".join(triplet),
                "unsigned_angle_deg": unsigned_angle,
                "unsigned_l2": unsigned_l2,
                "max_position_row_time_offset_s": float(np.nanmax([abs(sat_data[s]["position_row_time_offset_s"]) for s in triplet])),
                "used_position_fallback_count": int(sum(bool(sat_data[s]["used_position_fallback"]) for s in triplet)),
                "within_numerical_reason": bool((unsigned_angle <= PLANAR_ANGLE_TOL_DEG) and (unsigned_l2 <= PLANAR_UNSIGNED_L2_TOL)),
            })

    spherical_wf = event_normals.get("spherical", {})
    sats = spherical_wf.get("satellites", [])
    if spherical_wf.get("status") == "ok" and spherical_wf.get("method") == "fixed_radius_4sat" and len(sats) == 4:
        if all(canonical_sat_name(s) in param_key_map for s in sats):
            sat_data = build_event_sat_data_wolfram(event=event, sat_names=sats)

            py_fit = fixed_radius_sphere_normals(
                sat_data={s: {"t": sat_data[s]["t"], "r": sat_data[s]["r"]} for s in sats},
                v_kms=np.asarray(spherical_wf["velocity"], dtype=float),
            )

            wf_center = np.asarray(spherical_wf["center"], dtype=float)
            wf_radius = float(spherical_wf["r0"])
            wf_normals_map = {canonical_sat_name(k): np.asarray(v, dtype=float) for k, v in spherical_wf["normals"].items()}

            sat_angles = []
            sat_l2 = []
            for sat in sats:
                wf_sat_normal = wf_normals_map[canonical_sat_name(sat)]
                py_sat_normal = py_fit["normals"][sat]
                angle = unsigned_angle_deg(py_sat_normal, wf_sat_normal)
                ul2 = float(min(np.linalg.norm(py_sat_normal - wf_sat_normal), np.linalg.norm(py_sat_normal + wf_sat_normal)))
                sat_angles.append(angle)
                sat_l2.append(ul2)
                spherical_sat_rows.append({
                    "event": event,
                    "sat": sat,
                    "normal_unsigned_angle_deg": angle,
                    "normal_unsigned_l2": ul2,
                })

            center_l2 = float(np.linalg.norm(py_fit["center"] - wf_center))
            radius_abs_err = float(abs(py_fit["radius"] - wf_radius))

            spherical_rows.append({
                "event": event,
                "satellites": ", ".join(sats),
                "max_normal_unsigned_angle_deg": float(np.max(sat_angles)),
                "max_normal_unsigned_l2": float(np.max(sat_l2)),
                "center_l2_re": center_l2,
                "radius_abs_err_re": radius_abs_err,
                "max_position_row_time_offset_s": float(np.nanmax([abs(sat_data[s]["position_row_time_offset_s"]) for s in sats])),
                "used_position_fallback_count": int(sum(bool(sat_data[s]["used_position_fallback"]) for s in sats)),
                "within_numerical_reason": bool(
                    (np.max(sat_angles) <= SPHERE_MAX_NORMAL_ANGLE_TOL_DEG)
                    and (center_l2 <= SPHERE_CENTER_TOL_RE)
                    and (radius_abs_err <= SPHERE_RADIUS_TOL_RE)
                ),
            })

planar_validation_df = pd.DataFrame(planar_rows).sort_values("event").reset_index(drop=True)
spherical_validation_df = pd.DataFrame(spherical_rows).sort_values("event").reset_index(drop=True)
spherical_sat_validation_df = pd.DataFrame(spherical_sat_rows).sort_values(["event", "sat"]).reset_index(drop=True)

summary_df = pd.DataFrame([
    {
        "model": "planar",
        "checked_cases": len(planar_validation_df),
        "within_numerical_reason": int(planar_validation_df["within_numerical_reason"].sum()) if len(planar_validation_df) else 0,
    },
    {
        "model": "spherical_fixed_radius_4sat",
        "checked_cases": len(spherical_validation_df),
        "within_numerical_reason": int(spherical_validation_df["within_numerical_reason"].sum()) if len(spherical_validation_df) else 0,
    },
])

print("Planar validation")
display(planar_validation_df)
print("\nSpherical validation")
display(spherical_validation_df)
print("\nPer-satellite spherical normal comparison")
display(spherical_sat_validation_df)
print("\nSummary")
display(summary_df)


## Notes

- The planar model uses **3 satellites** and gives one front normal.
- The fixed-radius spherical model uses **4 satellites** and returns one normal per satellite.
- For real events, data quality and near-singular geometry matter; this notebook keeps only the core model logic.
